In [3]:
# ------------------ CALCULATE 2D DESCRIPTORS FROM SMILES ------------------

import pandas as pd

from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors


# ------------------ INPUT SMILES ------------------

smiles_text = """
O=C(CN1C=C(C2=C(C3=CC=C(Cl)C=C3Cl)C(=O)NC2=O)C2=CC=CC=C21)NCCOCCOCCNC1=CC=CC2=C1C(=O)N(C1CCC(=O)NC1=O)C2=O
O=C(CN1C=C(C2=C(C3=CC=C(Cl)C=C3Cl)C(=O)NC2=O)C2=CC=CC=C21)NCCCOCCCCOCCCNC1=CC=CC2=C1C(=O)N(C1CCC(=O)NC1=O)C2=O
Cc1c(c2ccc(CNC(C3CC(CN3C(C(C(C)(C)C)NC(COCCOCCNC(COc3cc4c(c(C5CCN(CC5)C(c5ccc(cc5)OC(F)(F)F)=O)ncn4)nc3)=O)=O)=O)O)=O)cc2)scn1
CC1=C(C2=CC=C(CNC(=O)[C@@H]3C[C@@H](O)CN3C(=O)[C@@H](NC(=O)COCCCOCCOC(=O)C[C@@H]3N=C(C4=CC=C(Cl)C=C4)C4=C(SC(C)=C4C)N4C(C)=NN=C34)C(C)(C)C)C=C2)SC=N1
CC1=C(C2=CC=C(CNC(=O)[C@@H]3C[C@@H](O)CN3C(=O)[C@@H](NC(=O)COCCCCCOCCNC(=O)C[C@@H]3N=C(C4=CC=C(Cl)C=C4)C4=C(SC(C)=C4C)N4C(C)=NN=C34)C(C)(C)C)C=C2)SC=N1
CC1=C(C2=CC=C([C@H](C)NC(=O)[C@@H]3C[C@@H](O)CN3C(=O)[C@@H](NC(=O)COCCCOCCOC(=O)C[C@@H]3N=C(C4=CC=C(Cl)C=C4)C4=C(SC(C)=C4C)N4C(C)=NN=C34)C(C)(C)C)C=C2)SC=N1
CC1=C(C2=CC=C([C@H](C)NC(=O)[C@@H]3C[C@@H](O)CN3C(=O)[C@@H](NC(=O)COCCCOCCNC(=O)C[C@@H]3N=C(C4=CC=C(Cl)C=C4)C4=C(SC(C)=C4C)N4C(C)=NN=C34)C(C)(C)C)C=C2)SC=N1
CC1=C(C2=CC=C(CNC(=O)[C@@H]3C[C@@H](O)CN3C(=O)[C@@H](NC(=O)COCCCCCOCCOC(=O)C[C@@H]3N=C(C4=CC=C(Cl)C=C4)C4=C(SC(C)=C4C)N4C(C)=NN=C34)C(C)(C)C)C=C2)SC=N1
O=C1CCC(C2=CC=C(NCC(=O)N3CCN(C(=O)C4CCN(C5=CC=C(NC(=O)C6=CN=C(OC7=CC=CC=C7)N=C6)C=C5)CC4)CC3)C=C2)C(=O)N1
CC1=C(C2=CC=C(CNC(=O)[C@@H]3C[C@@H](O)CN3C(=O)[C@@H](NC(=O)COCCOCCOCCOC(=O)C[C@@H]3N=C(C4=CC=C(Cl)C=C4)C4=C(SC(C)=C4C)N4C(C)=NN=C34)C(C)(C)C)C=C2)SC=N1
CC1=C(C2=CC=C(CNC(=O)[C@@H]3C[C@@H](O)CN3C(=O)[C@@H](NC(=O)COCCCOCCNC(=O)C[C@@H]3N=C(C4=CC=C(Cl)C=C4)C4=C(SC(C)=C4C)N4C(C)=NN=C34)C(C)(C)C)C=C2)SC=N1
O=C1CCC(N2C(=O)C3=CC=CC(N4CCN(C(=O)C5CCN(C6=CC=C(NC(=O)C7=CN=C(OC8=CC=CC=C8)N=C7)C=C6)CC5)CC4)=C3C2=O)C(=O)N1
O=C1CCC(N2C(=O)C3=CC=C(N4CCN(C(=O)C5CCN(C6=CC=C(NC(=O)C7=CN=C(OC8=CC=CC=C8)N=C7)C=C6)CC5)CC4)C=C3C2=O)C(=O)N1
CC1=C(C2=CC=C(CNC(=O)[C@@H]3C[C@@H](O)CN3C(=O)[C@@H](NC(=O)COCCOCCOCCNC(=O)C[C@@H]3N=C(C4=CC=C(Cl)C=C4)C4=C(SC(C)=C4C)N4C(C)=NN=C34)C(C)(C)C)C=C2)SC=N1
CN1C=CC=C1C(=O)N1CCN(C(=O)NC2=CC=C(N3CCC(C(=O)N4CCN(C5=CC=C6C(=O)N(C7CCC(=O)NC7=O)C(=O)C6=C5)CC4)CC3)C=C2)CC1
O=C1CCC(N2C(=O)C3=CC(F)=C(N4CCN(C(=O)C5CCN(C6=CC=C(NC(=O)C7=CN=C(OC8=CC=CC=C8)N=C7)C=C6)CC5)CC4)C=C3C2=O)C(=O)N1
COC1=CC(C2=CN(C)C(=O)C3=CN=CC=C23)=CC(OC)=C1CN1CCN(CCOCCOC2=CC(C3=C(C)N=CS3)=CC=C2CNC(=O)[C@@H]2C[C@@H](O)CN2C(=O)[C@@H](NC(=O)C2(F)CC2)C(C)(C)C)CC1
COC1=CC(C2=CN(C)C(=O)C3=CN=CC=C23)=CC(OC)=C1CN1CCN(CCOCCOCCOC2=CC(C3=C(C)N=CS3)=CC=C2CNC(=O)[C@@H]2C[C@@H](O)CN2C(=O)[C@@H](NC(=O)C2(F)CC2)C(C)(C)C)CC1
COC1=CC(C2=CN(C)C(=O)C3=CN=CC=C23)=CC(OC)=C1CN1CCN(CCCCCOC2=CC(C3=C(C)N=CS3)=CC=C2CNC(=O)[C@@H]2C[C@@H](O)CN2C(=O)[C@@H](NC(=O)C2(F)CC2)C(C)(C)C)CC1
CN1C=CC=C1C(=O)N1CCN(C(=O)NC2=CC=C(N3CCC(C(=O)N4CCN(C5=CC=CC6=C5C(=O)N(C5CCC(=O)NC5=O)C6=O)CC4)CC3)C=C2)CC1
"""

smiles_list = [
    line.strip()
    for line in smiles_text.strip().splitlines()
    if line.strip()
]

protac_names = [
    f"PROTAC-{i}"
    for i in range(1, len(smiles_list) + 1)
]


# ------------------ CALCULATION ------------------

results = []

for protac_name, smiles in zip(protac_names, smiles_list):

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        print(f"Could not read {protac_name}")
        continue

    # Explicitly add all hydrogens for total atom counting
    mol_with_h = Chem.AddHs(mol)

    # Average molecular weight; implicit hydrogens are included
    molecular_weight = Descriptors.MolWt(mol)

    # Exact molecular weight
    exact_mass = rdMolDescriptors.CalcExactMolWt(mol)

    # Atom counts
    total_atoms = mol_with_h.GetNumAtoms()
    heavy_atoms = mol.GetNumHeavyAtoms()
    hydrogen_atoms = total_atoms - heavy_atoms

    # Standard 2D descriptors
    rotatable_bonds = rdMolDescriptors.CalcNumRotatableBonds(mol)
    hbd = rdMolDescriptors.CalcNumHBD(mol)
    hba = rdMolDescriptors.CalcNumHBA(mol)
    ring_count = rdMolDescriptors.CalcNumRings(mol)

    # Standard RDKit TPSA
    tpsa = rdMolDescriptors.CalcTPSA(mol)

    # Optional TPSA including sulfur and phosphorus contributions
    tpsa_with_s_p = rdMolDescriptors.CalcTPSA(
        mol,
        includeSandP=True
    )

    # Wildman-Crippen calculated LogP and molar refractivity
    crippen_logp, molar_refractivity = (
        rdMolDescriptors.CalcCrippenDescriptors(mol)
    )

    # Labute approximate surface area
    labute_asa = rdMolDescriptors.CalcLabuteASA(mol)

    molecular_formula = rdMolDescriptors.CalcMolFormula(mol)

    results.append({
        "PROTAC": protac_name,
        "SMILES": smiles,
        "Molecular_formula": molecular_formula,
        "Molecular_weight_g_mol": molecular_weight,
        "Exact_mass_Da": exact_mass,
        "Total_atoms_with_H": total_atoms,
        "Hydrogen_atoms": hydrogen_atoms,
        "Heavy_atoms": heavy_atoms,
        "Rotatable_bonds": rotatable_bonds,
        "HBD": hbd,
        "HBA": hba,
        "Ring_count": ring_count,
        "TPSA_A2": tpsa,
        "TPSA_including_S_P_A2": tpsa_with_s_p,
        "Crippen_LogP": crippen_logp,
        "Molar_refractivity": molar_refractivity,
        "Labute_ASA_A2": labute_asa
    })


# ------------------ RESULTS ------------------

df = pd.DataFrame(results)

numeric_columns = df.select_dtypes(include="number").columns
df[numeric_columns] = df[numeric_columns].round(2)

display(df)

output_csv = "protac_2D_descriptors_from_smiles.csv"
df.to_csv(output_csv, index=False)

print(f"\nSaved: {output_csv}")
print(f"Successfully calculated: {len(df)} of {len(smiles_list)} molecules")

,PROTAC,SMILES,Molecular_formula,Molecular_weight_g_mol,Exact_mass_Da,Total_atoms_with_H,Hydrogen_atoms,Heavy_atoms,Rotatable_bonds,HBD,HBA,Ring_count,TPSA_A2,TPSA_including_S_P_A2,Crippen_LogP,Molar_refractivity,Labute_ASA_A2
0,PROTAC-1,O=C(CN1C=C(C2=C(C3=CC=C(Cl)C=C3Cl)C(=O)NC2=O)C...,C39H34Cl2N6O9,801.64,800.18,90,34,56,15,4,11,7,194.24,194.24,3.18,204.61,328.80
1,PROTAC-2,O=C(CN1C=C(C2=C(C3=CC=C(Cl)C=C3Cl)C(=O)NC2=O)C...,C43H42Cl2N6O9,857.75,856.24,102,42,60,19,4,11,7,194.24,194.24,4.74,223.08,354.26
2,PROTAC-3,Cc1c(c2ccc(CNC(C3CC(CN3C(C(C(C)(C)C)NC(COCCOCC...,C50H58F3N9O10S,1034.13,1033.40,131,58,73,20,4,15,7,236.63,264.87,4.71,260.12,422.73
3,PROTAC-4,CC1=C(C2=CC=C(CNC(=O)[C@@H]3C[C@@H](O)CN3C(=O)...,C48H57ClN8O8S2,973.62,972.34,124,57,67,18,3,15,7,199.46,255.94,6.40,256.75,402.67
4,PROTAC-5,CC1=C(C2=CC=C(CNC(=O)[C@@H]3C[C@@H](O)CN3C(=O)...,C50H62ClN9O7S2,1000.69,999.39,131,62,69,20,4,14,7,202.26,258.74,6.75,268.04,415.83
5,PROTAC-6,CC1=C(C2=CC=C([C@H](C)NC(=O)[C@@H]3C[C@@H](O)C...,C49H59ClN8O8S2,987.65,986.36,127,59,68,18,3,15,7,199.46,255.94,6.96,261.47,409.03
6,PROTAC-7,CC1=C(C2=CC=C([C@H](C)NC(=O)[C@@H]3C[C@@H](O)C...,C49H60ClN9O7S2,986.66,985.37,128,60,68,18,4,14,7,202.26,258.74,6.53,263.52,409.47
7,PROTAC-8,CC1=C(C2=CC=C(CNC(=O)[C@@H]3C[C@@H](O)CN3C(=O)...,C50H61ClN8O8S2,1001.67,1000.37,130,61,69,20,3,15,7,199.46,255.94,7.18,265.99,415.40
8,PROTAC-9,O=C1CCC(C2=CC=C(NCC(=O)N3CCN(C(=O)C4CCN(C5=CC=...,C40H42N8O6,730.83,730.32,96,42,54,10,3,10,7,166.17,166.17,4.04,201.10,312.44
9,PROTAC-10,CC1=C(C2=CC=C(CNC(=O)[C@@H]3C[C@@H](O)CN3C(=O)...,C49H59ClN8O9S2,1003.64,1002.35,128,59,69,20,3,16,7,208.69,265.17,6.02,262.96,414.15



Saved: protac_2D_descriptors_from_smiles.csv
Successfully calculated: 20 of 20 molecules
